# Customer Churn Prediction & Recommendation System

# Phase 7 – Model Training & Evaluation

## Objectives

- Train multiple Machine Learning models
- Compare their performance
- Evaluate using different metrics
- Select the best model
- Save the best model

### Import libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import joblib

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt

### Import custom module

In [2]:
import sys

sys.path.append("..")

from src.preprocessing.data_loader import load_data

### Load Dataset

In [3]:
df = load_data("../data/processed/Telco_Customer_Churn_Cleaned.csv")

df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Separate feature and target

In [4]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

print(X.shape)
print(y.shape)

(7043, 19)
(7043,)


### Encode target

In [5]:
target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y)

print(target_encoder.classes_)

['No' 'Yes']


### Identify feature type

In [6]:
categorical_columns = X.select_dtypes(include=["object", "string"]).columns.tolist()

numerical_columns = X.select_dtypes(exclude=["object", "string"]).columns.tolist()

print("Categorical Columns")
print(categorical_columns)

print()

print("Numerical Columns")
print(numerical_columns)

Categorical Columns
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Numerical Columns
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


### preprocessing pipeline 

In [7]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            StandardScaler(),
            numerical_columns
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        )

    ]

)

### Test-train Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

### Transform data

In [9]:
X_train = preprocessor.fit_transform(X_train)

X_test = preprocessor.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(5634, 45)
(1409, 45)


### Create models dictionary

In [11]:
models = {

    "Logistic Regression":

        LogisticRegression(),

    "Decision Tree":

        DecisionTreeClassifier(random_state=42),

    "Random Forest":

        RandomForestClassifier(random_state=42),

    "Gradient Boosting":

        GradientBoostingClassifier(random_state=42),

    "XGBoost":

        XGBClassifier(
            eval_metric="logloss",
            random_state=42
        ),

    "LightGBM":

        LGBMClassifier(random_state=42),

    "CatBoost":

        CatBoostClassifier(
            verbose=0,
            random_state=42
        )

}

### Train every Model

In [12]:
results = []

trained_models = {}

for name, model in models.items():

    print("=" * 50)

    print(name)

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(y_test, predictions)

    recall = recall_score(y_test, predictions)

    f1 = f1_score(y_test, predictions)

    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1

    })

    trained_models[name] = model

    print("Accuracy :", round(accuracy,4))

Logistic Regression
Accuracy : 0.8055
Decision Tree
Accuracy : 0.7289
Random Forest
Accuracy : 0.7779
Gradient Boosting
Accuracy : 0.8062
XGBoost
Accuracy : 0.7842
LightGBM
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001322 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 669
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265353 -> initscore=-1.018328
[LightGBM] [Info] Start training from score -1.018328
Accuracy : 0.7977
CatBoost
Accuracy : 0.797


### Compare Model

In [14]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(

    by="Accuracy",

    ascending=False

)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
3,Gradient Boosting,0.806246,0.673540,0.524064,0.589474
0,Logistic Regression,0.805536,0.657233,0.558824,0.604046
5,LightGBM,0.797729,0.639498,0.545455,0.588745
6,CatBoost,0.797019,0.642857,0.529412,0.580645
4,XGBoost,0.784244,0.605422,0.537433,0.569405
2,Random Forest,0.777857,0.603390,0.475936,0.532138
1,Decision Tree,0.728886,0.489637,0.505348,0.497368


# 📊 Model Performance Analysis & Best Model Selection

## Model Comparison Summary

The performance of seven Machine Learning classification algorithms was evaluated using the following metrics:

- Accuracy
- Precision
- Recall
- F1 Score

| Model | Accuracy | Precision | Recall | F1 Score |
|-------|---------:|----------:|-------:|---------:|
| Gradient Boosting | **80.62%** | **67.35%** | 52.41% | 58.95% |
| Logistic Regression | 80.55% | 65.72% | **55.88%** | **60.40%** |
| LightGBM | 79.77% | 63.95% | 54.55% | 58.87% |
| CatBoost | 79.70% | 64.29% | 52.94% | 58.06% |
| XGBoost | 78.42% | 60.54% | 53.74% | 56.94% |
| Random Forest | 77.79% | 60.34% | 47.59% | 53.21% |
| Decision Tree | 72.89% | 48.96% | 50.53% | 49.74% |

---

## Best Model Selection

Although **Gradient Boosting** achieved the highest accuracy (80.62%), **Logistic Regression** was selected as the final model for this project.

### Reasons for Selecting Logistic Regression

- Achieved the **highest F1 Score (60.40%)**, providing the best balance between Precision and Recall.
- Produced the **highest Recall (55.88%)**, meaning it correctly identified more customers who are likely to churn.
- Offers excellent interpretability, making it easier to explain predictions to business stakeholders.
- Requires less computational power and provides faster predictions during deployment.
- Well suited for binary classification problems such as Customer Churn Prediction.
- A lightweight model that integrates efficiently with REST APIs and web applications.

---

## Final Selected Model

**Model:** Logistic Regression

### Final Performance

- **Accuracy:** 80.55%
- **Precision:** 65.72%
- **Recall:** 55.88%
- **F1 Score:** 60.40%

Based on the evaluation metrics and business requirements, **Logistic Regression** was selected as the production model for predicting customer churn.

## Business Perspective

In customer churn prediction, identifying customers who are likely to leave is more valuable than simply achieving the highest accuracy. Therefore, Recall and F1 Score are considered more important evaluation metrics than Accuracy alone.

Selecting Logistic Regression helps the company identify more at-risk customers, enabling timely retention campaigns while maintaining strong overall predictive performance.